In [ ]:
import openpyxlimport pandas as pdimport numpy as npfrom torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSamplerfrom tqdm import tqdmdata=pd.read_excel('./data/datos.xlsx')data['parsed'] = data['parsed'].astype(str)data['parsed'] = data['parsed'].str.split(r'\$\%\&') #separar las frases con los separadoresdata['text'] = data['text'].str.strip() #eliminar espacios rarostexts = data.textnsarc = data.nsarcnast = data.nastinessdata

In [ ]:
import torchfrom transformers import BertTokenizer, BertModel# Cargar el tokenizador y el modelo de BERTtokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)# Pon el modelo en modo de evaluaciónmodel.eval()

In [ ]:
tokens = [tokenizer.encode_plus(texto, add_special_tokens=True, max_length=256, pad_to_max_length=True, truncation=True) for texto in texts]input_ids = [tokens['input_ids'] for tokens in tokens]attention_masks = [tokens['attention_mask'] for tokens in tokens]

In [ ]:
inputs = torch.stack([torch.tensor(np.array(id)) for id in input_ids])masks = torch.stack([torch.tensor(np.array(mask)) for mask in attention_masks])

In [ ]:
datas = TensorDataset(inputs,masks)dataloader = DataLoader(datas,batch_size=32)

In [ ]:
device = torch.device("cpu")output_tensor = []for batch in tqdm(dataloader, desc="Processing"):# Add batch to GPU  batch = tuple(t.to(device) for t in batch)  with torch.no_grad():# Unpack the inputs from our dataloader    b_input_ids, b_input_mask = batch    output = model(b_input_ids, b_input_mask)#print(len(output[0][0]))#print(output)    output = output.last_hidden_state.detach().to('cpu')    output_tensor.extend(output)

In [ ]:
# Stack tensors along a new dimension (dim=0)output_tensor = torch.stack(output_tensor, dim=0)print(output_tensor.size())

In [ ]:
#Calculate the meanmean_tensor = torch.mean(output_tensor, dim=1)print(mean_tensor.size())

In [ ]:
# Specify the file pathfile_path = "./data/mean_tensor.pt"# Save the tensor to the filetorch.save(mean_tensor, file_path)

In [ ]:
import torchnsarc_tensor = torch.tensor(nsarc.values)# Specify the file pathfile_path = "./data/nsarc_tensor.pt"# Save the tensor to the filetorch.save(nsarc_tensor, file_path)

In [ ]:
import torchnast_tensor = torch.tensor(nast.values)# Specify the file pathfile_path = "./data/nast_tensor.pt"# Save the tensor to the filetorch.save(nast_tensor, file_path)